# LeetCode 901: Online Stock Span

**Difficulty**: Medium  
**Topics**: Stack, Design, Monotonic Stack, Data Stream  
**Link**: [LeetCode Problem](https://leetcode.com/problems/online-stock-span/)

---

## Problem Statement

Design an algorithm that collects daily price quotes for some stock and returns the **span** of that stock's price for the current day.

The **span** of the stock's price in one day is the **maximum number of consecutive days** (starting from that day and going backward) for which the stock price was **less than or equal to** the price of that day.

### What is Stock Span?

For a given day, the span is:
- **1** if today's price is lower than yesterday's
- **Number of consecutive previous days** (including today) where price ≤ today's price

### Visual Example

```
Day:    1    2    3    4    5    6    7
Price: 100  80   60   70   60   75   85
Span:   1    1    1    2    1    4    6

Explanation:
Day 1: span = 1 (only today)
Day 2: span = 1 (80 < 100, so only today)
Day 3: span = 1 (60 < 80, so only today)
Day 4: span = 2 (70 > 60, includes days 3-4)
Day 5: span = 1 (60 < 70, so only today)
Day 6: span = 4 (75 > 60, 70, 60, includes days 3-6)
Day 7: span = 6 (85 > all previous, includes days 2-7)
```

### Constraints

- `1 <= price <= 10^5`
- At most `10^4` calls will be made to `next`

### Interface

```python
class StockSpanner:
    def __init__(self):
        # Initialize data structure
    
    def next(self, price: int) -> int:
        # Return the span for the current price
```

---

## Approach 1: Brute Force (Store All Prices)

### Intuition

Store all prices in a list. For each new price, scan backward counting consecutive days where price ≤ current price.

### Algorithm

```
1. Store all prices in a list
2. For each new price:
     count = 1
     Look backward from the previous day
     While previous_price <= current_price:
         count++
         Move to earlier day
     Return count
```

### Complexity

- **Time**: O(n) per call in worst case (scan all previous prices)
- **Space**: O(n) to store all prices

In [ ]:
class StockSpanner_BruteForce:
    """
    Brute force approach.
    Time: O(n) per call, Space: O(n)
    """
    def __init__(self):
        self.prices = []
    
    def next(self, price: int) -> int:
        self.prices.append(price)
        span = 1
        
        # Scan backward
        i = len(self.prices) - 2
        while i >= 0 and self.prices[i] <= price:
            span += 1
            i -= 1
        
        return span

# Test
spanner = StockSpanner_BruteForce()
prices = [100, 80, 60, 70, 60, 75, 85]
result = [spanner.next(p) for p in prices]
print(f"Prices: {prices}")
print(f"Spans:  {result}")
print(f"Expected: [1, 1, 1, 2, 1, 4, 6]")

### Why Brute Force Is Slow

For an increasing sequence like `[10, 20, 30, 40, 50]`:
- Day 1: check 0 prices
- Day 2: check 1 price
- Day 3: check 2 prices
- Day 4: check 3 prices
- Day 5: check 4 prices

Total: 0 + 1 + 2 + 3 + 4 = 10 operations for 5 prices = O(n²) overall

---

## Approach 2: Monotonic Stack (Optimal)

### Intuition

The key insight: **We don't need to check every previous price individually.**

If we already know that:
- Day 3 has span 1
- Day 4 has span 2 (includes days 3-4)

And Day 5's price is greater than Day 4's price, then:
- Day 5's span includes Day 4's entire span (2 days)
- Plus Day 5 itself (1 day)
- Total: 2 + 1 = 3 days

**We can merge spans instead of counting individual days!**

### The Monotonic Stack Trick

Maintain a stack of `(price, span)` pairs where:
- Prices are in **decreasing order** (monotonic decreasing)
- When a higher price arrives, pop all smaller prices and **add their spans**

### Visual Example

For prices `[100, 80, 60, 70, 60, 75, 85]`:

```
Day 1: price=100
  Stack: [(100, 1)]
  Span: 1

Day 2: price=80
  80 < 100, don't pop
  Stack: [(100, 1), (80, 1)]
  Span: 1

Day 3: price=60
  60 < 80, don't pop
  Stack: [(100, 1), (80, 1), (60, 1)]
  Span: 1

Day 4: price=70
  70 > 60, pop (60, 1), add span: 1 + 1 = 2
  70 < 80, stop popping
  Stack: [(100, 1), (80, 1), (70, 2)]
  Span: 2

Day 5: price=60
  60 < 70, don't pop
  Stack: [(100, 1), (80, 1), (70, 2), (60, 1)]
  Span: 1

Day 6: price=75
  75 > 60, pop (60, 1), add span: 1 + 1 = 2
  75 > 70, pop (70, 2), add span: 2 + 2 = 4
  75 < 80, stop popping
  Stack: [(100, 1), (80, 1), (75, 4)]
  Span: 4

Day 7: price=85
  85 > 75, pop (75, 4), add span: 1 + 4 = 5
  85 > 80, pop (80, 1), add span: 5 + 1 = 6
  85 < 100, stop popping
  Stack: [(100, 1), (85, 6)]
  Span: 6
```

### Why This Works

When we pop `(price_i, span_i)`, it means:
- Current price is greater than `price_i`
- `price_i` already counted `span_i` consecutive days
- We can **inherit** those `span_i` days instead of recounting them

### Algorithm

```
1. Initialize empty stack
2. For each new price:
     span = 1
     While stack not empty AND stack.top.price <= current_price:
         (old_price, old_span) = stack.pop()
         span += old_span
     stack.push((current_price, span))
     return span
```

### Complexity

- **Time**: O(1) amortized per call
  - Each price is pushed once and popped at most once
  - Total operations over n calls: at most 2n
- **Space**: O(n) for the stack

In [ ]:
class StockSpanner:
    """
    Optimal solution using monotonic stack.
    Time: O(1) amortized, Space: O(n)
    """
    def __init__(self):
        self.stack = []  # Stack of (price, span) pairs
    
    def next(self, price: int) -> int:
        span = 1
        
        # Pop all prices <= current price and accumulate their spans
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        
        # Push current price with its span
        self.stack.append((price, span))
        return span

# Test
spanner = StockSpanner()
prices = [100, 80, 60, 70, 60, 75, 85]
result = [spanner.next(p) for p in prices]
print(f"Prices: {prices}")
print(f"Spans:  {result}")
print(f"Expected: [1, 1, 1, 2, 1, 4, 6]")

### Detailed Walkthrough

Let's trace through the algorithm step by step:

In [ ]:
class StockSpanner_Verbose:
    """Verbose version showing each step."""
    def __init__(self):
        self.stack = []
        self.day = 0
    
    def next(self, price: int) -> int:
        self.day += 1
        span = 1
        
        print(f"\nDay {self.day}: price = {price}")
        print(f"  Stack before: {self.stack}")
        
        # Pop and accumulate
        popped = []
        while self.stack and self.stack[-1][0] <= price:
            old_price, old_span = self.stack.pop()
            popped.append((old_price, old_span))
            span += old_span
            print(f"  Pop ({old_price}, {old_span}), span now = {span}")
        
        if not popped:
            print(f"  No pops (current price not greater than stack top)")
        
        self.stack.append((price, span))
        print(f"  Push ({price}, {span})")
        print(f"  Stack after: {self.stack}")
        print(f"  Return span: {span}")
        
        return span

# Test
spanner = StockSpanner_Verbose()
prices = [100, 80, 60, 70, 60, 75, 85]
print("Processing stock prices...")
result = [spanner.next(p) for p in prices]
print(f"\nFinal result: {result}")

---

## Edge Cases

In [ ]:
# Edge case 1: All increasing prices
print("All increasing prices:")
spanner = StockSpanner()
result = [spanner.next(p) for p in [10, 20, 30, 40, 50]]
print(f"Prices: [10, 20, 30, 40, 50]")
print(f"Spans:  {result}")
print(f"Expected: [1, 2, 3, 4, 5]\n")

# Edge case 2: All decreasing prices
print("All decreasing prices:")
spanner = StockSpanner()
result = [spanner.next(p) for p in [50, 40, 30, 20, 10]]
print(f"Prices: [50, 40, 30, 20, 10]")
print(f"Spans:  {result}")
print(f"Expected: [1, 1, 1, 1, 1]\n")

# Edge case 3: All same prices
print("All same prices:")
spanner = StockSpanner()
result = [spanner.next(p) for p in [50, 50, 50, 50]]
print(f"Prices: [50, 50, 50, 50]")
print(f"Spans:  {result}")
print(f"Expected: [1, 2, 3, 4]\n")

# Edge case 4: Single price
print("Single price:")
spanner = StockSpanner()
result = [spanner.next(100)]
print(f"Prices: [100]")
print(f"Spans:  {result}")
print(f"Expected: [1]\n")

# Edge case 5: Zigzag pattern
print("Zigzag pattern:")
spanner = StockSpanner()
result = [spanner.next(p) for p in [10, 20, 10, 20, 10, 20]]
print(f"Prices: [10, 20, 10, 20, 10, 20]")
print(f"Spans:  {result}")
print(f"Expected: [1, 2, 1, 2, 1, 2]")

---

## Comparison of Approaches

| Approach | Time per Call | Space | Notes |
|----------|--------------|-------|-------|
| Brute Force | O(n) worst case | O(n) | Simple but slow for long sequences |
| Monotonic Stack | O(1) amortized | O(n) | Optimal, each price processed once |

### Performance Example

For 10,000 calls with increasing prices:
- **Brute Force**: 1 + 2 + 3 + ... + 10,000 = 50,005,000 operations
- **Monotonic Stack**: ~20,000 operations (each price pushed/popped once)

**Speedup**: ~2,500x faster!

---

## Common Mistakes

### Mistake 1: Storing Only Prices Without Spans

**Wrong**:
```python
self.stack.append(price)  # Lost span information!
```

**Right**:
```python
self.stack.append((price, span))  # Store both
```

### Mistake 2: Using < Instead of <=

**Wrong**:
```python
while self.stack and self.stack[-1][0] < price:  # Misses equal prices
```

**Right**:
```python
while self.stack and self.stack[-1][0] <= price:  # Includes equal prices
```

The problem says "less than or equal to", so equal prices should be included in the span.

### Mistake 3: Forgetting to Initialize Span to 1

**Wrong**:
```python
span = 0  # Doesn't count today!
```

**Right**:
```python
span = 1  # Start with today
```

### Mistake 4: Not Accumulating Spans Correctly

**Wrong**:
```python
span = self.stack.pop()[1]  # Overwrites instead of accumulates
```

**Right**:
```python
span += self.stack.pop()[1]  # Accumulate all popped spans
```

---

## Why Amortized O(1)?

### The Concern

The `while` loop looks like it could take O(n) time in a single call.

### The Reality

Each price can only be:
- **Pushed once**: When it arrives
- **Popped at most once**: When a higher price arrives

Over n calls:
- Total pushes: n
- Total pops: at most n
- Total operations: 2n

**Average per call**: 2n / n = 2 = O(1)

### Example Count

For prices `[100, 80, 60, 70, 60, 75, 85]`:

```
Call | Pushes | Pops | Total Ops
-----|--------|------|----------
  1  |   1    |  0   |    1
  2  |   1    |  0   |    1
  3  |   1    |  0   |    1
  4  |   1    |  1   |    2
  5  |   1    |  0   |    1
  6  |   1    |  2   |    3
  7  |   1    |  2   |    3
-----|--------|------|----------
Total|   7    |  5   |   12
```

12 operations for 7 calls = 1.7 operations per call on average

---

## Related Problems

- [496. Next Greater Element I](https://leetcode.com/problems/next-greater-element-i/) - Basic next greater pattern
- [503. Next Greater Element II](https://leetcode.com/problems/next-greater-element-ii/) - Circular array variant
- [739. Daily Temperatures](https://leetcode.com/problems/daily-temperatures/) - Return distance to next greater
- [84. Largest Rectangle in Histogram](https://leetcode.com/problems/largest-rectangle-in-histogram/) - Uses similar monotonic stack pattern
- [42. Trapping Rain Water](https://leetcode.com/problems/trapping-rain-water/) - Another monotonic stack application

---

## Key Takeaways

1. **Stock span** = consecutive days going backward where price ≤ today's price
2. **Monotonic stack** stores `(price, span)` pairs in decreasing order
3. **Span accumulation**: When popping, add the popped span to current span
4. **Why it works**: We inherit previously computed spans instead of recounting
5. **Time complexity**: O(1) amortized - each price pushed/popped at most once
6. **Use <=** not < because equal prices count in the span
7. **Pattern**: This is a "previous smaller/equal" monotonic stack problem
8. **Real-world use**: Stock market analysis, trend detection, moving averages

### The Core Insight

Instead of counting individual days, we **merge spans** of consecutive days. This transforms an O(n) operation into O(1) amortized.